[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/slm-development/blob/main/tutorial/01_small_language_models_what_they_are_and_why_they_matter/03_setting_up_your_environment.ipynb)

# Setting Up Your Training Environment

| | |
|---|---|
| **Domain** | GenAI |
| **Module** | Small Language Models: What They Are and Why They Matter |
| **Difficulty** | Beginner |
| **Estimated Time** | 25 minutes |
| **Prerequisites** | Basic Python programming knowledge |

> **💡 If you are running this in Google Colab, your environment is already set up!** The install cell below handles everything. You can skip the local setup sections and go straight to the diagnostic.

---

## Learning Objectives

By the end of this lesson, you will be able to:

- Install the complete SLM training stack with pinned versions
- Verify GPU availability — or configure a CPU fallback — using a diagnostic script
- Understand why dependency isolation prevents silent training failures
- Authenticate with Hugging Face Hub for accessing gated models

---

## ⚙️ Install Everything — Run This Cell First

This single cell installs the **complete SLM training stack** with pinned versions. It works on Colab, local Jupyter, or any Python 3.10+ environment.

In [ ]:
# ───────────────────────────────────────────────────────────────
# COMPLETE SLM TRAINING STACK — PINNED VERSIONS
# This is the single source of truth for all dependencies.
# Last verified: 2025-06
# ───────────────────────────────────────────────────────────────

!pip install -q \
    transformers==4.41.2 \
    torch==2.3.0 \
    datasets==2.19.2 \
    peft==0.11.1 \
    accelerate==0.30.1 \
    sentencepiece==0.2.0 \
    huggingface_hub==0.23.0 \
    evaluate==0.4.2 \
    rouge-score==0.1.2 \
    safetensors==0.4.3

print("✅ All packages installed. Running diagnostic...\n")

---

## 🟢 Core Concepts

### How Conflicting Dependencies Break Training Runs

A Python environment is a container for packages and their exact versions. Without isolation, installing two projects side by side almost guarantees a conflict. For SLM training, the failure mode is subtle: **PyTorch silently falls back to CPU** when a mismatched CUDA version breaks the GPU bridge. You won't see an error — you'll just see your training run take 40× longer than expected.

Think of a virtual environment like a separate toolbox for each project. The tools in one box never interfere with another box.

### Compatibility Matrix (Single Source of Truth)

| Component | Pinned Version | Last verified |
|---|---|---|
| Python | 3.10+ (3.11 recommended) | 2025-06 |
| PyTorch | 2.3.0 | 2025-06 |
| Hugging Face Transformers | 4.41.2 | 2025-06 |
| PEFT | 0.11.1 | 2025-06 |
| Datasets (HF) | 2.19.2 | 2025-06 |
| Accelerate | 0.30.1 | 2025-06 |

---

## 🔷 Environment Diagnostic

Run the cell below to verify that every package version is correct and check GPU availability.

In [ ]:
"""
Environment diagnostic — confirms GPU availability and correct package versions.
All five version lines must show ✅.
"""

import sys
import torch
import transformers
import datasets
import peft
from packaging.version import Version

EXPECTED = {
    "torch": "2.3.0",
    "transformers": "4.41.2",
    "datasets": "2.19.2",
    "peft": "0.11.1",
}

print("=" * 50)
print("  SLM Training Environment Diagnostic")
print("=" * 50)
print()

# Python version
py_major, py_minor = sys.version_info.major, sys.version_info.minor
py_ok = py_major == 3 and py_minor >= 10
print(f"{'\u2705' if py_ok else '\u274c'}  Python {py_major}.{py_minor} (need 3.10+)")

# Package versions — using proper semantic version comparison
packages = {
    "torch": torch.__version__.split("+")[0],  # strip +cu121 suffix
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "peft": peft.__version__,
}

all_ok = py_ok
for name, installed in packages.items():
    expected = EXPECTED[name]
    try:
        match = Version(installed) >= Version(expected)
    except Exception:
        match = installed.startswith(expected)
    status = "\u2705" if match else "\u274c"
    if not match:
        all_ok = False
    print(f"{status}  {name} {installed} (expected >= {expected})")

# GPU check
print()
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"\u2705  GPU detected: {device_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("\u26a0\ufe0f  No GPU detected \u2014 CPU fallback active.")
    print("   All code in this course works on CPU (training will be slower).")

print()
if all_ok:
    print("\u2705  All checks passed! Your environment is ready.")
else:
    print("\u274c  Some checks failed. Re-run the install cell above.")
print("=" * 50)

---

## 🔷 Hugging Face Hub Authentication

Several models in later modules (starting Module 4) require a Hugging Face account.

1. Create a free account at [huggingface.co](https://huggingface.co)
2. Navigate to **Settings → Access Tokens** and create a token with **read** scope
3. Run the cell below to authenticate

In [ ]:
# Authenticate with Hugging Face Hub
# This stores your token securely — it is NEVER saved in the notebook.

from huggingface_hub import login

# On Colab: this opens a login widget
# On local: paste your token at the prompt
login()

print("\n\u2705 Authentication successful!")
print("\u26a0\ufe0f  Never commit your HF token to version control.")

> **Important:** Never commit your HF token to version control. In CI pipelines, use environment variables (`HF_TOKEN`).

---

## 🟢 Local Setup Guide (if not using Colab)

If you prefer to work locally instead of Colab, follow these steps:

### macOS / Linux
```bash
python3.11 -m venv slm_env
source slm_env/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
```

### Windows (CMD)
```cmd
py -3.11 -m venv slm_env
slm_env\Scripts\activate.bat
pip install --upgrade pip
pip install -r requirements.txt
```

### Windows (PowerShell)
```powershell
py -3.11 -m venv slm_env
Set-ExecutionPolicy -Scope CurrentUser RemoteSigned
slm_env\Scripts\Activate.ps1
pip install --upgrade pip
pip install -r requirements.txt
```

> The `requirements.txt` at the repo root contains all pinned versions.

---

## 🧪 Hands-On Exercise

**Goal:** Produce a verified environment snapshot.

In [ ]:
# Export your exact environment for reproducibility
import subprocess
import sys
from datetime import datetime

# Capture pip freeze output
result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True, text=True
)

packages = result.stdout.strip().split("\n")
key_packages = [p for p in packages if any(
    name in p.lower() for name in ["torch", "transformers", "datasets", "peft", "accelerate"]
)]

print(f"Environment snapshot — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Python: {sys.version.split()[0]}")
print(f"\nKey packages ({len(key_packages)}):")
for pkg in sorted(key_packages):
    print(f"  {pkg}")

print(f"\nTotal packages installed: {len(packages)}")
print("\n\u2705 Snapshot complete! Any teammate can reproduce your environment.")

---

## ❓ Concept Check

### Q1: Which Python and PyTorch versions does this course pin?

- **A)** Python 3.10 + PyTorch 2.0
- **B)** Python 3.10+ + PyTorch 2.3.0 — tested together for GPU compatibility
- **C)** Python 3.12 + PyTorch 2.4
- **D)** Any Python 3.x + PyTorch 1.x

<details><summary>🔑 Answer</summary>

**B.** PyTorch GPU support depends on a three-way match: Python version, PyTorch version, and CUDA toolkit.
</details>

### Q2: You run the diagnostic and see `\u274c peft 0.9.0 (expected 0.11.1)`. What's the fix?

- **A)** `pip install peft` (installs latest)
- **B)** Delete and recreate the virtual environment
- **C)** `pip install --force-reinstall peft==0.11.1`
- **D)** Upgrade torch and peft simultaneously

<details><summary>🔑 Answer</summary>

**C.** `--force-reinstall` replaces the installed version with the exact pinned version without touching other packages.
</details>

---

## Summary

- **Isolate dependencies first.** Version conflicts cause silent GPU fallback — the hardest failure to debug.
- **Pin every version.** The compatibility matrix is your source of truth.
- **Run the diagnostic before writing training code.** A \u2705 on every line means your environment is reproducible.
- **HF Hub authentication is needed from Module 4 onward.** Set it up now.

---

## References & Credits

- PyTorch Installation Selector: [pytorch.org/get-started](https://pytorch.org/get-started/locally/) *(Last verified: 2025-06)*
- Hugging Face Hub tokens: [docs](https://huggingface.co/docs/hub/security-tokens) *(Last verified: 2025-06)*
- PEFT library (Apache 2.0): [github.com/huggingface/peft](https://github.com/huggingface/peft)